# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll print all available record sets and their fields using their `@id`, name, and type.

In [ ]:
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
overview = {}
for rs in record_sets:
    print(f"Record set @id: {rs.id}")
    print(f"  Name: {rs.name}  (Type: {rs.type})")
    print("  Fields:")
    overview[rs.id] = []
    for field in rs.fields:
        print(f"    - Field @id: {field.id}\n      Name: {field.name} (dataType: {field.data_type})")
        overview[rs.id].append(field.id)
    print()

## 3. Data Extraction
Load data from each available record set into a DataFrame for analysis. Each record set is referenced by its `@id`.

In [ ]:
# List of all record set @id's
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame()
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()} (Num records: {len(df)})\n")

if not record_set_ids:
    print("No record sets found in the dataset.")
else:
    # Choose the first non-empty record set for next examples.
    for rid in record_set_ids:
        if not dataframes[rid].empty:
            example_record_set_id = rid
            break
    else:
        example_record_set_id = record_set_ids[0]
    print(f"Head of example record set ({example_record_set_id}):")
    display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll select a numeric field (by its `@id`) and a group field (by its `@id`) from the overview above. **Update the field IDs below as appropriate for your selected record set!**

In [ ]:
# --- Specify field @id's for EDA below --- #
# Example assignment (update according to your dataset record set and field ids)

record_set_id = example_record_set_id    # Use an identified and loaded record set
df = dataframes[record_set_id].copy()

# After running the previous cell, examine column names to fill these variables appropriately
# Example: numeric_field = 'cr:coverage_percent', group_field = 'cr:sample_id'
if not df.empty:
    print(f"Available columns: {df.columns.tolist()}")
    # Try to auto-detect a numeric field
    # Common heuristics: choose first float/integer column
    num_candidate = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            num_candidate = col
            break
    if num_candidate is None and len(df.columns) > 0:
        # Try to convert first column to numeric if possible
        try:
            df[df.columns[0]] = pd.to_numeric(df[df.columns[0]], errors='coerce')
            if pd.api.types.is_numeric_dtype(df[df.columns[0]]):
                num_candidate = df.columns[0]
        except Exception:
            pass
    # Try to find a group field (likely string/categorical)
    group_candidate = None
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]) and df[col].nunique() > 1 and df[col].nunique() < len(df) / 2:
            group_candidate = col
            break

    numeric_field = num_candidate if num_candidate is not None else df.columns[0]
    group_field = group_candidate if group_candidate is not None else None

    threshold = df[numeric_field].mean() if numeric_field is not None else 0
    print(f"\nUsing numeric field: {numeric_field} (threshold: {threshold:.2f})")
    if group_field:
        print(f"Using group field: {group_field}")

    if numeric_field is not None:
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
else:
    print("No available records to analyze in the selected record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using `matplotlib` or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and numeric_field is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(12, 4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Used `mlcroissant` to load dataset metadata and explore available record sets and fields by their `@id`.
- Loaded and previewed records from each record set as pandas DataFrames.
- Selected a numeric field for EDA, filtered and normalized its values, and grouped by a categorical field where available.
- Visualized the distribution and grouping of key variables.

**Next steps:** This notebook can be adapted to more detailed analysis, modeling, or integration with other FAIR data tools.